# Final Evaluation

**Task IDs:** T-41  
**Purpose:** Evaluate the best model ONCE on the held-out test set.
Confusion matrix heatmap, ROC curve, PR curve, metric table.

Findings recorded in `03-evaluation/Metrics Reference.md` (Obsidian vault).

In [ ]:
import json
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix, roc_curve, precision_recall_curve

from src.config import get_path, load_config
from src.evaluation.metrics import compute_metrics, format_metrics

warnings.filterwarnings("ignore", category=FutureWarning)
cfg = load_config()
models_dir = get_path(cfg, "models_dir")
processed_dir = get_path(cfg, "processed_dir")

with open(models_dir / "threshold.json") as f:
    threshold_data = json.load(f)
threshold = threshold_data["threshold"]
print(f"Using threshold: {threshold:.4f} ({threshold_data['policy']})")

In [ ]:
model = joblib.load(models_dir / "best.joblib")
test = pd.read_parquet(processed_dir / "test.parquet")
X_test = test.drop(columns=["Class"])
y_test = test["Class"]

y_proba = model.predict_proba(X_test)[:, 1]
y_pred = (y_proba >= threshold).astype(int)

metrics = compute_metrics(y_test, y_pred, y_proba)
print(format_metrics(metrics))

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Legitimate", "Fraud"],
            yticklabels=["Legitimate", "Fraud"], ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Confusion Matrix (Test Set)")
plt.tight_layout()
plt.savefig("../reports/figures/evaluation/confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_proba)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(fpr, tpr, color="#3b82f6", lw=2)
axes[0].plot([0, 1], [0, 1], "--", color="#8892a8")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title(f"ROC Curve (AUC={metrics['roc_auc']:.4f})")

precision, recall, _ = precision_recall_curve(y_test, y_proba)
axes[1].plot(recall, precision, color="#ef4444", lw=2)
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title(f"PR Curve (AP={metrics['pr_auc']:.4f})")

plt.tight_layout()
plt.savefig("../reports/figures/evaluation/roc_pr_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## Threshold Tuning

Sweep thresholds on the validation set to find the optimal F1 and
cost-weighted thresholds (per ADR-004).

In [ ]:
from src.evaluation.threshold import sweep_thresholds, best_f1_threshold, cost_weighted_threshold

val = pd.read_parquet(processed_dir / "val.parquet")
X_val = val.drop(columns=["Class"])
y_val = val["Class"]
y_val_proba = model.predict_proba(X_val)[:, 1]

sweep = sweep_thresholds(y_val, y_val_proba)
f1_thresh = best_f1_threshold(sweep)
cost_thresh = cost_weighted_threshold(y_val, y_val_proba, cost_fp=1, cost_fn=50)

print(f"F1-maximising threshold:    {f1_thresh:.4f}")
print(f"Cost-weighted threshold:    {cost_thresh:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(sweep["thresholds"], sweep["f1"], color="#3b82f6")
axes[0].axvline(f1_thresh, color="#ef4444", linestyle="--", label=f"F1 max @ {f1_thresh:.3f}")
axes[0].set_xlabel("Threshold")
axes[0].set_ylabel("F1 Score")
axes[0].set_title("F1 vs Threshold")
axes[0].legend()

costs = []
for t in sweep["thresholds"]:
    yp = (y_val_proba >= t).astype(int)
    fp = ((yp == 1) & (y_val == 0)).sum()
    fn = ((yp == 0) & (y_val == 1)).sum()
    costs.append(fp + 50 * fn)
axes[1].plot(sweep["thresholds"], costs, color="#f59e0b")
axes[1].axvline(cost_thresh, color="#ef4444", linestyle="--", label=f"Cost min @ {cost_thresh:.3f}")
axes[1].set_xlabel("Threshold")
axes[1].set_ylabel("Total Cost (FP + 50*FN)")
axes[1].set_title("Cost vs Threshold")
axes[1].legend()

plt.tight_layout()
plt.savefig("../reports/figures/threshold/threshold_tuning.png", dpi=150, bbox_inches="tight")
plt.show()

## SHAP Analysis

Global feature importance + top-6 dependence plots.

In [ ]:
import shap

if hasattr(model, "named_steps") and "model" in model.named_steps:
    estimator = model.named_steps["model"]
elif hasattr(model, "steps"):
    estimator = model.steps[-1][1]
else:
    estimator = model

feature_names = list(X_test.columns)
explainer = shap.TreeExplainer(estimator)
shap_values = explainer(np.asarray(X_test))
shap_values.feature_names = feature_names

plt.figure(figsize=(10, 8))
shap.plots.beeswarm(shap_values, max_display=20, show=False)
plt.tight_layout()
plt.savefig("../reports/figures/shap/summary_plot.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
mean_abs_shap = np.abs(shap_values.values).mean(axis=0)
top6_idx = np.argsort(mean_abs_shap)[-6:][::-1]
top6_features = [feature_names[i] for i in top6_idx]
print("Top 6 features by mean |SHAP|:", top6_features)

for feat in top6_features:
    idx = feature_names.index(feat)
    plt.figure(figsize=(8, 6))
    shap.plots.scatter(shap_values[:, idx], color=shap_values, show=False)
    plt.tight_layout()
    plt.savefig(f"../reports/figures/shap/dependence_{feat}.png", dpi=150, bbox_inches="tight")
    plt.show()

## Error Analysis

Profile false positives and false negatives.

In [ ]:
from src.evaluation.error_analysis import error_breakdown, plot_error_amounts

errors = error_breakdown(test, y_test, y_pred)
print(f"False positives: {len(errors['false_positives'])}")
print(f"False negatives: {len(errors['false_negatives'])}")
print(f"True positives:  {len(errors['true_positives'])}")

plot_error_amounts(
    errors["true_positives"],
    errors["false_positives"],
    errors["false_negatives"],
    "../reports/figures/error_analysis/amounts_by_error_type.png",
)

fp_profile = errors["false_positives"][["Amount", "log_amount"]].describe()
fn_profile = errors["false_negatives"][["Amount", "log_amount"]].describe()
print("\nFalse positive Amount profile:")
print(fp_profile)
print("\nFalse negative Amount profile:")
print(fn_profile)

## Summary

Record final test-set metrics, threshold choice, SHAP findings, and error
patterns in the Obsidian vault:
- `03-evaluation/Metrics Reference.md`
- `03-evaluation/Explainability SHAP.md`
- `03-evaluation/Error Analysis.md`
- `03-evaluation/Threshold Tuning.md`